# Week 07 — BBO capstone driver

Round 7. A ±0.02 offset from the W6 point on every coordinate — an order of magnitude larger than W6's step, on the reasoning that if 0.003 was too small, 0.02 may clear the noise floor.

**The right diagnosis, the wrong response.** The correct move once a step size is shown to be uninformative is to change what the round is *for*, not to scale the same step and spend another round. Two rounds are now committed to the same neighbourhood.

**Error worth flagging:** F8's submitted vector this round is identical to W6's, so that one query is a pure repeat rather than an offset. Unintended at the time — though a deliberate repeat is exactly the reproduction test this project never otherwise ran.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 7
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 7
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: '±0.02 offset from W6',
    2: '±0.02 offset from W6',
    3: '±0.02 offset from W6',
    4: '±0.02 offset from W6',
    5: '±0.02 offset from W6',
    6: '±0.02 offset from W6',
    7: '±0.02 offset from W6',
    8: 'repeat of W6 (unintended)',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 6. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — ±0.02 from W6

The `identical_to_W6` flag catches the F8 duplicate.

In [ ]:
proposals = {
    1: np.array([0.669237, 0.752052]),
    2: np.array([0.355232, 0.449399]),
    3: np.array([0.110375, 0.775016, 0.516551]),
    4: np.array([0.184472, 0.064553, 0.017918, 0.903826]),
    5: np.array([0.308106, 0.731274, 0.091098, 0.662517]),
    6: np.array([0.069721, 0.531532, 0.952453, 0.826305, 0.228097]),
    7: np.array([0.965268, 0.154515, 0.588291, 0.807859, 0.098927, 0.700438]),
    8: np.array([0.018683, 0.255785, 0.161747, 0.328043, 0.783074, 0.214943, 0.955801, 0.074683]),
}

prev = {fid: np.array(bbo.HISTORY[6][fid][0]) for fid in bbo.FUNC_IDS}
pd.DataFrame([dict(func=f"F{fid}",
                   step=round(float(np.linalg.norm(proposals[fid]-prev[fid])), 5),
                   identical_to_W6=bool(np.allclose(proposals[fid], prev[fid])),
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.048421, 0.137247],
    2: [0.048699, 0.137537],
    3: [0.489658, 0.159666, 0.895963],
    4: [0.569121, 0.44372, 0.402671, 0.283008],
    5: [0.275418, 0.703962, 0.058314, 0.62974],
    6: [0.448901, 0.916285, 0.331657, 0.205493, 0.612847],
    7: [0.659037, 0.842715, 0.27649, 0.501628, 0.787126, 0.394208],
    8: [0.015899, 0.258569, 0.158963, 0.325259, 0.785858, 0.212159, 0.958585, 0.071899],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 7 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: ...,
#     2: ...,
#     3: ...,
#     4: ...,
#     5: ...,
#     6: ...,
#     7: ...,
#     8: ...,
# }
#
# Returns for this round are not in the working record.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
